# 01 – Data Collection and First Exploration

**Objective:** download ATP match data, check its quality, and set a first baseline for prediction.

**Key results**
- **Dataset:** 79,299 ATP matches from January 2000 to June 2026.
- **Baseline:** the better-ranked player wins 65.4% of matches.
- **Missing rankings:** only 0.76% of winners have no ranking, and 92% of those cases are in Davis Cup matches.

## 1. Setup

- `pandas`: works with tables (DataFrames).
- `pathlib.Path`: handles file paths on any operating system.
- `urllib.request`: downloads files from the internet.

In [1]:
import pandas as pd
from pathlib import Path
import urllib.request

# Folder where the raw CSV files will be saved
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Data will be saved in:", RAW_DIR)

Data will be saved in: ../data/raw


## 2. Download the data

The data was created by Jeff Sackmann and is split into one CSV file per year. His original repository is no longer online, so this notebook uses an archive mirror.

The analysis starts in 2000 because bookmaker odds, needed in a later phase, are only available from around 2001. Files already downloaded are skipped.

In [2]:
# Jeff Sackmann's original repo is no longer online, so we use an archive mirror of it
BASE_URL = "https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/main/atp/"

for year in range(2000, 2027):
    filename = f"atp_matches_{year}.csv"
    path = RAW_DIR / filename
    if path.exists():
        continue
    try:
        urllib.request.urlretrieve(BASE_URL + filename, path)
        print("Downloaded", filename)
    except Exception as e:
        print("Not available:", filename, "-", e)

# Also download the players file (names, hand, height, country, birth date)
players_path = RAW_DIR / "atp_players.csv"
if not players_path.exists():
    urllib.request.urlretrieve(BASE_URL + "atp_players.csv", players_path)
print("Done.")

Done.


## 3. Combine the yearly files

Each yearly file is read and stacked into a single DataFrame with `pd.concat`.

In [3]:
files = sorted(RAW_DIR.glob("atp_matches_*.csv"))
matches = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Files loaded:", len(files))
print("Matches:", len(matches))
print("Columns:", matches.shape[1])

Files loaded: 27
Matches: 79299
Columns: 49


## 4. First look

Each row is one match, with separate columns for the winner and the loser.

**Note for modelling:** because the winner always sits in the `winner_` columns, this format can't be used directly to train a model. This is addressed during feature engineering.

In [4]:
pd.set_option("display.max_columns", None)  # show all columns
matches.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2000-301,Auckland,Hard,32,A,20000110,1,103163,1.0,NaN,Tommy Haas,R,188.0,GER,21.7,101543,NaN,NaN,Jeff Tarango,L,180.0,USA,31.1,7-5 4-6 7-5,3,R32,108.0,18.0,4.0,96.0,49.0,39.0,28.0,17.0,3.0,5.0,7.0,8.0,106.0,55.0,39.0,29.0,17.0,4.0,7.0,11.0,1612.0,63.0,595.0
1,2000-301,Auckland,Hard,32,A,20000110,2,102607,NaN,Q,Juan Balcells,R,190.0,ESP,24.5,102644,NaN,NaN,Franco Squillari,L,183.0,ARG,24.3,7-5 7-5,3,R32,85.0,5.0,3.0,76.0,52.0,39.0,13.0,12.0,5.0,6.0,5.0,10.0,74.0,32.0,25.0,18.0,12.0,3.0,6.0,211.0,157.0,49.0,723.0
2,2000-301,Auckland,Hard,32,A,20000110,3,103252,NaN,NaN,Alberto Martin,R,175.0,ESP,21.3,102238,NaN,NaN,Alberto Berasategui,R,173.0,ESP,26.5,6-3 6-1,3,R32,56.0,0.0,0.0,55.0,35.0,25.0,12.0,8.0,1.0,1.0,0.0,6.0,56.0,33.0,20.0,7.0,8.0,7.0,11.0,48.0,726.0,59.0,649.0
3,2000-301,Auckland,Hard,32,A,20000110,4,103507,7.0,NaN,Juan Carlos Ferrero,R,183.0,ESP,19.9,103819,NaN,NaN,Roger Federer,R,185.0,SUI,18.4,6-4 6-4,3,R32,68.0,5.0,1.0,53.0,28.0,26.0,15.0,10.0,0.0,0.0,11.0,2.0,70.0,43.0,29.0,14.0,10.0,6.0,8.0,45.0,768.0,61.0,616.0
4,2000-301,Auckland,Hard,32,A,20000110,5,102103,NaN,Q,Michael Sell,R,180.0,USA,27.3,102765,4.0,NaN,Nicolas Escude,R,185.0,FRA,23.7,0-6 7-6(7) 6-1,3,R32,115.0,1.0,2.0,98.0,66.0,39.0,14.0,13.0,6.0,11.0,8.0,8.0,92.0,46.0,34.0,18.0,12.0,5.0,9.0,167.0,219.0,34.0,873.0


In [5]:
matches.tail()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
79294,2026-M-DC-2026-WG2-PO-RSA-MNE-01,Davis Cup WG2 PO: RSA vs MNE,Hard,4,D,20260207,4,202475,NaN,NaN,Philip Henning,R,NaN,RSA,25.2,210188,NaN,NaN,Petar Jovanovic,U,NaN,MNE,22.5,6-3 6-2,3,RR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,317.0,160.0,1679.0,2.0
79295,2026-M-DC-2026-WG2-PO-RSA-MNE-01,Davis Cup WG2 PO: RSA vs MNE,Hard,4,D,20260207,5,209278,NaN,NaN,Alec Beckley,R,183.0,RSA,24.6,210648,NaN,NaN,Aleksa Krivokapic,U,NaN,MNE,22.7,7-6(3) 6-3,3,RR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,783.0,35.0,NaN,NaN
79296,2026-M-DC-2026-WG2-PO-THA-PUR-01,Davis Cup WG2 PO: THA vs PUR,Hard,4,D,20260207,1,210382,NaN,NaN,Maximus Jones,R,185.0,THA,21.4,214425,NaN,NaN,Yannik Alvarez,U,NaN,PUR,17.3,6-1 6-2,3,RR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,304.0,172.0,NaN,NaN
79297,2026-M-DC-2026-WG2-PO-THA-PUR-01,Davis Cup WG2 PO: THA vs PUR,Hard,4,D,20260207,2,208937,NaN,NaN,Kasidit Samrej,R,191.0,THA,25.0,211670,NaN,NaN,Marko Mesarovic,U,NaN,USA,NaN,6-2 6-1,3,RR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,395.0,120.0,1307.0,6.0
79298,2026-M-DC-2026-WG2-PO-THA-PUR-01,Davis Cup WG2 PO: THA vs PUR,Hard,4,D,20260207,4,210687,NaN,NaN,Pawit Sornlaksup,U,NaN,THA,26.4,211670,NaN,NaN,Marko Mesarovic,U,NaN,USA,NaN,2-6 6-4 [11-9],3,RR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,703.0,46.0,1307.0,6.0


## 5. Dataset structure

| Group | Example columns |
|---|---|
| Tournament | `tourney_name`, `surface`, `tourney_level`, `tourney_date`, `round`, `best_of` |
| Players | `winner_name`, `winner_rank`, `winner_age`, `winner_ht`, `winner_hand`, and the same for `loser_` |
| Match statistics | `w_ace`, `w_svpt`, `w_1stWon`, and the same with `l_` |

**Data leakage risk:** match statistics are only known after a match is played. They can't be used to predict that same match, only to build features for later matches.

In [6]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79299 entries, 0 to 79298
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          79299 non-null  object 
 1   tourney_name        79299 non-null  object 
 2   surface             79246 non-null  object 
 3   draw_size           79299 non-null  int64  
 4   tourney_level       79299 non-null  object 
 5   tourney_date        79299 non-null  int64  
 6   match_num           79299 non-null  int64  
 7   winner_id           79299 non-null  int64  
 8   winner_seed         32990 non-null  float64
 9   winner_entry        10191 non-null  object 
 10  winner_name         79299 non-null  object 
 11  winner_hand         79299 non-null  object 
 12  winner_ht           77756 non-null  float64
 13  winner_ioc          79299 non-null  object 
 14  winner_age          79287 non-null  float64
 15  loser_id            79299 non-null  int64  
 16  lose

## 6. Matches by surface

In [7]:
# Matches per surface
matches["surface"].value_counts()

surface
Hard      43461
Clay      25746
Grass      8000
Carpet     2039
Name: count, dtype: int64

## 7. Baseline: always pick the better-ranked player

A lower rank number means a better player. Any model must beat the accuracy of this simple rule to be useful.

In [8]:
# How often does the better-ranked player (lower rank number) win?
# This is our first BASELINE: the score any model must beat.
valid = matches.dropna(subset=["winner_rank", "loser_rank"])
favourite_wins = (valid["winner_rank"] < valid["loser_rank"]).mean()
print(f"Better-ranked player wins {favourite_wins:.1%} of matches")

Better-ranked player wins 65.4% of matches


### Is the favourite more reliable on some surfaces?

In [9]:
clay = valid[valid["surface"] == "Clay"]
hard = valid[valid["surface"] == "Hard"]
grass = valid[valid["surface"] == "Grass"]

clay_pct = (clay["winner_rank"] < clay["loser_rank"]).mean()
hard_pct = (hard["winner_rank"] < hard["loser_rank"]).mean()
grass_pct = (grass["winner_rank"] < grass["loser_rank"]).mean()

print(f"Better-ranked player wins {clay_pct:.1%} on clay, {hard_pct:.1%} on hard, {grass_pct:.1%} on grass")

Better-ranked player wins 64.8% on clay, 65.9% on hard, 65.1% on grass


**Finding:** the differences are small: 64.8% on clay, 65.9% on hard and 65.1% on grass. Ranking alone predicts about equally well on all three surfaces.

## 8. Most match wins since 2000

In [10]:
matches["winner_name"].value_counts().head(10)

winner_name
Roger Federer      1250
Novak Djokovic     1188
Rafael Nadal       1091
Andy Murray         748
David Ferrer        740
Tomas Berdych       643
Richard Gasquet     613
Andy Roddick        612
Marin Cilic         611
Stan Wawrinka       592
Name: count, dtype: int64

**Finding:** Federer (1,250), Djokovic (1,188) and Nadal (1,091) lead by a wide margin. These totals only count matches from 2000 onwards.

## 9. Missing rankings

In [11]:
matches[["winner_rank", "loser_rank"]].isna().sum()

winner_rank     599
loser_rank     1553
dtype: int64

In [12]:
missing = matches[matches["winner_rank"].isna()]
missing[["tourney_name", "tourney_date", "tourney_level", "winner_name"]].head(20)

,tourney_name,tourney_date,tourney_level,winner_name
75,Munich,20000501,A,Martin Damm Sr
877,Miami Masters,20000320,M,Mardy Fish
1482,Cincinnati Masters,20000807,M,Martin Damm Sr
1745,Umag,20000717,A,Martin Damm Sr
1759,Umag,20000717,A,Martin Damm Sr
1775,s Hertogenbosch,20000619,A,Martin Damm Sr
1790,s Hertogenbosch,20000619,A,Martin Damm Sr
1797,s Hertogenbosch,20000619,A,Martin Damm Sr
1950,Copenhagen,20000228,A,Martin Damm Sr
1963,Copenhagen,20000228,A,Martin Damm Sr


In [13]:
missing["tourney_level"].value_counts()

tourney_level
D    553
A     37
G      6
M      3
Name: count, dtype: int64

**Findings**
- **Size of the problem:** `winner_rank` is missing in 599 matches (0.76%) and `loser_rank` in 1,553 (1.96%).
- **Main source:** 553 of the 599 missing winner ranks (92%) are in Davis Cup matches (level `D`). National teams select their own players, so some of them probably have no ATP ranking.
- **Likely data gap:** some missing ranks don't fit that explanation. Martin Damm Sr has no rank in any of his wins, including ATP and Grand Slam matches. A separate player named Martin Damm appears in 2024–2026, so a missing rank doesn't always mean an unranked player.
- **Implication:** players should be identified by `winner_id` and `loser_id`, because names aren't reliable identifiers.